# Session 4 — Build Your Own Mini RAG 🛠️

Welcome back! In **Session 3** we built the *vocabulary* — tokens, embeddings, RAG, prompts.
Today we turn every one of those concepts into runnable Python.

By the end of this notebook you will:

- 💬 Talk to an LLM over the Azure OpenAI API
- 🔤 See exactly how text becomes tokens
- 🌡️ Feel what *temperature* does
- 🎭 Use system prompts, few-shot, and chain-of-thought
- 🧲 Turn text into embeddings and measure meaning with cosine similarity
- ✂️ Chunk a long document
- 🗄️ Store + search vectors in **ChromaDB**
- 🔄 Wire it all up into a working **mini RAG pipeline**
- 📦 See the same code organized into a tinkerable Python project (`mini_rag/`)

> 🧠 **Mental model from last session:** *LLMs are pattern completion engines. RAG hands them the right pages just before they answer.* Today is about building that "hand them the right pages" part.


## 1 — Setup 🔌

Before we run anything, install the libraries and load our Azure OpenAI credentials.

> 💡 If you've already run `pip install -r requirements.txt` in a terminal, you can skip the install cell.


In [ ]:
# Run this once. The leading ! lets us run shell commands inside Jupyter.
!pip install -q openai chromadb tiktoken numpy python-dotenv


Copy `.env.example` to `.env` and fill in your **Azure OpenAI** keys:

```
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_API_VERSION=2024-06-01
AZURE_OPENAI_CHAT_DEPLOYMENT=gpt-4o-mini
AZURE_OPENAI_EMBED_DEPLOYMENT=text-embedding-3-small
```

The `python-dotenv` package will load them as environment variables.


In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()  # reads .env into os.environ

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

CHAT_MODEL = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]
EMBED_MODEL = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]

print("✅ Client ready. Chat model:", CHAT_MODEL, "| Embed model:", EMBED_MODEL)


## 2 — Anatomy of a Chat Call 💬

A chat call is just a list of `messages`. Each message has a **role** (`system`, `user`, or `assistant`) and **content**.

The model reads the whole list and produces the next assistant message.


In [ ]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a friendly tutor. Keep answers under 2 sentences."},
        {"role": "user", "content": "What is Python in one line?"},
    ],
)

print(response.choices[0].message.content)
print("---")
print("Tokens used:", response.usage.total_tokens,
      "(prompt:", response.usage.prompt_tokens,
      "+ completion:", response.usage.completion_tokens, ")")


> 🧠 **Notice the `usage` block.** Every call tells you exactly how many tokens it consumed. This is what you pay for. Keep an eye on it as your prompts grow.

## 3 — Tokens: the "atoms of language" 🔤

From Session 3: **LLMs don't read words, they read tokens.** Common words = 1 token; longer/unusual words get split.

We can use the `tiktoken` library to *see* the split. (It uses OpenAI's tokenizer, which closely matches what Azure OpenAI does.)


In [ ]:
import tiktoken

# `cl100k_base` is the tokenizer used by GPT-4 / GPT-4o / GPT-4o-mini.
enc = tiktoken.get_encoding("cl100k_base")

samples = [
    "The cat sat on the mat",
    "Unbelievable!",
    "AI",
    "anthropic",
    "Tokenization is surprisingly important.",
]

for text in samples:
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{text!r:50s} -> {len(ids)} tokens: {pieces}")


> 📏 **Rule of thumb:** 1 token ≈ 4 characters ≈ ¾ of a word.

Try it yourself — change the strings above. Notice how `"anthropic"` (lowercase, less common) splits into pieces, while `"AI"` is a single token.


## 4 — Next-Token Prediction 🎰

From the slides: an LLM does **one thing** — predict the next token, then the next, then the next.

We can *see* this by **streaming** a response. Instead of waiting for the full answer, we get tokens as they're generated.


In [ ]:
stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Write a haiku about debugging."}],
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()  # final newline


> 👀 You just watched a "pattern completion engine" at work, one token at a time. That's *all* it's doing — and yet here we are.

## 5 — Temperature: the Creativity Dial 🌡️

- `temperature=0` → almost always picks the most likely token. Repeatable, "safe."
- `temperature=1.0+` → flatter probabilities, more variety.

Let's run the *same* prompt three times at each temperature and compare.


In [ ]:
def generate(prompt, temperature):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=40,
    )
    return response.choices[0].message.content.strip()

prompt = "Give me a one-sentence tagline for a coffee shop."

print("=== temperature = 0 (precise) ===")
for _ in range(3):
    print("•", generate(prompt, 0))

print("\n=== temperature = 1.0 (creative) ===")
for _ in range(3):
    print("•", generate(prompt, 1.0))


> 💡 **Practical tip:** factual / business apps → keep temperature low (0–0.3). Creative writing → 0.7+. **For RAG, almost always use 0 — you want grounded answers, not creative ones.**

## 6 — System Prompts 🎬

The **system prompt** sets the model's persona, rules, and tone. It's the single highest-leverage thing you can change.

Same `user` message, two different system prompts:


In [ ]:
def with_system(system, user):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return response.choices[0].message.content

user_msg = "How do I fix a leaky faucet?"

print("🏴‍☠️ Pirate Helper:")
print(with_system("You are a pirate. Always speak like a pirate.", user_msg))
print("\n🤵 Formal Tutor:")
print(with_system("You are a formal expert. Be concise and step-by-step.", user_msg))


## 7 — Few-Shot: Teaching by Example 📝

You don't need fine-tuning to teach a pattern. Show the model 2–3 examples in the prompt and it generalizes.


In [ ]:
few_shot_prompt = '''Convert informal text to professional email language.

Input: "Hey, can u fix this ASAP?"
Output: "Hello, could you please address this at your earliest convenience?"

Input: "thx for the help!"
Output: "Thank you for your assistance."

Input: "gonna need that report by tmrw"
Output:'''

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0,
)
print(response.choices[0].message.content.strip())


> 🧠 The model learned the *pattern* (informal → professional) from just two examples. Adding more examples almost always improves consistency.

## 8 — Chain-of-Thought: "Think Step by Step" 🧮

For reasoning problems, asking the model to **show its work** dramatically improves accuracy. The intermediate tokens become "scratch paper" the model can use.


In [ ]:
problem = (
    "I want to buy 3 shirts at $25 each. There's a 20% discount. "
    "I have $60. Can I afford all 3? Answer with just yes or no."
)

# Without CoT
plain = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": problem}],
    temperature=0,
).choices[0].message.content.strip()

# With CoT — same problem, just one extra instruction
cot_problem = problem.replace("Answer with just yes or no.", "Think step by step, then end with 'Answer: yes' or 'Answer: no'.")
cot = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": cot_problem}],
    temperature=0,
).choices[0].message.content.strip()

print("WITHOUT chain-of-thought:")
print(plain)
print("\n--- WITH chain-of-thought ---")
print(cot)


## 9 — The Limitation: No Private Data 🚧

Watch this. We'll ask the model about a **fictional** company's refund policy. It has no way of knowing — but it might *make something up* anyway. This is the hallucination problem from Slide 13.


In [ ]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "user", "content": "What is Acme Corp's refund policy? Be specific."}
    ],
    temperature=0,
)
print(response.choices[0].message.content)


> 😬 Either it confidently invents a policy, or it admits it doesn't know. Either way, it can't help your customers without **your data**.
>
> **This is exactly the problem RAG solves.** The rest of the notebook builds the fix.


## 10 — Embeddings: Text → Numbers That Capture Meaning 🧲

An **embedding** is a long list of numbers (usually ~1500 of them) that represents the *meaning* of a piece of text. Similar meanings → similar numbers.


In [ ]:
words = ["king", "queen", "banana"]

response = client.embeddings.create(model=EMBED_MODEL, input=words)
vectors = [item.embedding for item in response.data]

for word, vec in zip(words, vectors):
    print(f"{word:8s} → length={len(vec)}, first 8 dims: {[round(x, 3) for x in vec[:8]]}")


> 🔢 Each word becomes a vector of ~1500 numbers. The numbers themselves are meaningless to us — but the *distances between them* tell us how similar two pieces of text are. That's the next cell.

## 11 — Cosine Similarity: How Close Is Close? 📐

Two vectors pointing the same direction → similar (score ≈ 1).
At right angles → unrelated (≈ 0). Opposite → opposite meaning (≈ -1).

The math is just `dot(a, b) / (|a| * |b|)`. NumPy makes it a one-liner.


In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "How do I reset my password?",
    "I forgot my login credentials",
    "Steps to recover account access",
    "What's the weather today?",
    "The cat sat on the mat",
    "A feline rested on the rug",
]

embs = [item.embedding for item in client.embeddings.create(model=EMBED_MODEL, input=sentences).data]

print(f"{'A':40s} | {'B':40s} | score")
print("-" * 95)
pairs = [(0, 1), (0, 2), (0, 3), (4, 5), (4, 3)]
for i, j in pairs:
    print(f"{sentences[i]:40s} | {sentences[j]:40s} | {cosine(embs[i], embs[j]):.3f}")


> 🎯 The cat/feline example is the magic moment: **0 words in common**, yet a high similarity score. Embeddings capture *meaning*, not surface text.

## 12 — Chunking: Splitting Documents into Pieces ✂️

We can't embed an entire 100-page doc as one vector — meaning gets diluted, and the chunk wouldn't fit in the LLM's context anyway.

Instead, split into ~500-character chunks with a small **overlap** so we don't cut sentences in half. Let's write `chunk_text` from scratch.


In [ ]:
def chunk_text(text: str, size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into chunks of `size` chars with `overlap` shared chars."""
    chunks = []
    step = size - overlap
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        if start + size >= len(text):
            break
        start += step
    return chunks


sample = (
    "The refund policy requires customers to submit a request within 30 days of purchase. "
    "Refunds are processed within 5 business days to the original payment method. "
    "Items must be returned in their original packaging. "
    "Digital goods are non-refundable except in the case of technical defects."
)

print(f"Total length: {len(sample)} chars\n")

# Try with overlap
chunks_with_overlap = chunk_text(sample, size=120, overlap=30)
for i, c in enumerate(chunks_with_overlap):
    print(f"Chunk {i} ({len(c)} chars): {c!r}")


> 🔗 **Overlap matters.** Set `overlap=0` and re-run — you'll see sentences get sliced mid-thought. With overlap, both neighboring chunks share enough context that retrieval still works at the seams.

> 📏 **Rule of thumb:** 10–20% overlap. So 500-char chunks → 50–100 char overlap.


## 13 — ChromaDB: A Real Vector Database 🗄️

We *could* build our own search using NumPy and the `cosine` function above (and that's exactly what's happening under the hood). But once you have thousands of chunks, you want a real vector database to do it efficiently.

**ChromaDB** is open-source, runs locally, no setup needed. Perfect for our mini RAG.


In [ ]:
import chromadb

# A persistent client stores everything in ./chroma_demo/ on disk.
chroma_client = chromadb.PersistentClient(path="./chroma_demo")

# Reset the demo collection so this cell is re-runnable.
try:
    chroma_client.delete_collection("notebook_demo")
except Exception:
    pass

collection = chroma_client.create_collection("notebook_demo")

# A few short docs about completely different topics.
docs = [
    "Acme Corp offers full refunds within 30 days of purchase with a receipt.",
    "Our office hours are 9am to 5pm Pacific Time, Monday through Friday.",
    "The capital of France is Paris.",
    "Python is a high-level, interpreted programming language.",
]

doc_embeddings = [item.embedding for item in client.embeddings.create(model=EMBED_MODEL, input=docs).data]

collection.add(
    ids=[f"doc-{i}" for i in range(len(docs))],
    documents=docs,
    embeddings=doc_embeddings,
)

print(f"Stored {collection.count()} docs.")


In [ ]:
# Now query — by MEANING, not keywords.
query = "Can I get my money back?"
[query_embedding] = [item.embedding for item in client.embeddings.create(model=EMBED_MODEL, input=[query]).data]

results = collection.query(query_embeddings=[query_embedding], n_results=2)

print(f"Query: {query}\n")
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"  (distance={dist:.3f})  {doc}")


> 🔍 **Notice:** the query "Can I get my money back?" doesn't share any words with "Acme Corp offers full refunds..." — but it ranks first. That's vector search.

## 14 — Putting It Together: a Mini RAG 🔄

Now we have every piece. Let's wire them up:

1. **Chunk** a small "company handbook" defined inline.
2. **Embed** every chunk.
3. **Store** in Chroma.
4. **Embed** the user's question.
5. **Search** for the top-3 most similar chunks.
6. **Build** a prompt that includes those chunks.
7. **Ask** the LLM — it now has the context it was missing in 9.


In [ ]:
# Our tiny "company handbook" — pretend this came from internal docs.
company_handbook = '''
REFUND POLICY
Acme Corp offers full refunds within 30 days of purchase. Customers must provide
a receipt or order number. Digital goods are non-refundable except in cases of
proven technical defects. Refunds are processed to the original payment method
within 5 business days.

PTO POLICY
Full-time employees receive 20 days of paid time off per year, accrued monthly.
Unused PTO does not roll over past December 31. New hires must complete a
90-day probation period before PTO accrual begins.

IT SUPPORT
For password resets, employees should email it-help@acme.example or visit the
self-service portal at help.acme.example. Phone support is available Monday
through Friday, 8am to 6pm Pacific Time.
'''

# 1. Chunk
handbook_chunks = chunk_text(company_handbook.strip(), size=300, overlap=50)
print(f"Chunked into {len(handbook_chunks)} pieces.\n")

# 2. Embed
chunk_embeddings = [item.embedding for item in client.embeddings.create(model=EMBED_MODEL, input=handbook_chunks).data]

# 3. Store
try:
    chroma_client.delete_collection("rag_demo")
except Exception:
    pass
rag_collection = chroma_client.create_collection("rag_demo")
rag_collection.add(
    ids=[f"hb-{i}" for i in range(len(handbook_chunks))],
    documents=handbook_chunks,
    embeddings=chunk_embeddings,
)
print(f"Stored {rag_collection.count()} chunks in the vector DB.")


In [ ]:
def rag_ask(question, top_k=3):
    # 4. Embed the question
    [q_emb] = [item.embedding for item in client.embeddings.create(model=EMBED_MODEL, input=[question]).data]

    # 5. Search
    hits = rag_collection.query(query_embeddings=[q_emb], n_results=top_k)
    context = "\n\n---\n\n".join(hits["documents"][0])

    # 6. Build the grounded prompt (lifted from Slide 28)
    prompt = f"""Using ONLY this context, answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

    # 7. Ask the LLM
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers strictly from the provided context."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


# Re-ask the question that failed in 9 — but now grounded in our handbook.
print("Q: What is Acme Corp's refund policy?")
print("A:", rag_ask("What is Acme Corp's refund policy?"))
print()
print("Q: How many PTO days do new hires get on their first day?")
print("A:", rag_ask("How many PTO days do new hires get on their first day?"))
print()
print("Q: Who won the 2026 Super Bowl?")
print("A:", rag_ask("Who won the 2026 Super Bowl?"))


> 🎉 **That's RAG.** Compare the first answer with 9 — same model, same question, but now it answers correctly because we handed it the relevant page.
>
> The third question ("Super Bowl") tests our grounding instruction — the model should say "I don't know" because that info isn't in our context.


## 15 — Tour of the Project 📦

Everything we just wrote lives in a small Python package next to this notebook: **`mini_rag/`**. Same logic, just organized so you can extend it.

```
session4/
├── session4.ipynb            ← this notebook
├── mini_rag/
│   ├── config.py             ← Azure OpenAI client setup
│   ├── chunk.py              ← chunk_text() — same as 12
│   ├── embed.py              ← embed_texts() — wraps the embeddings API
│   ├── store.py              ← ChromaDB wrapper — same as 13
│   ├── ingest.py             ← walks corpus/, chunks, embeds, stores
│   └── rag.py                ← ask() — same as 14, packaged up
├── corpus/                   ← drop your .md / .txt files here
└── main.py                   ← CLI: `python main.py ingest` / `ask "..."`
```

Let's use it.


In [ ]:
# Drop a sample file into corpus/ and re-run this cell to try it out.
# (You can also do this from the command line with `python main.py ingest`.)

from pathlib import Path

# Make sure we have something to index.
sample_path = Path("corpus/handbook.md")
sample_path.write_text(company_handbook)
print("Wrote", sample_path)


In [ ]:
from mini_rag import ingest, ask

ingest("corpus")


In [ ]:
print(ask("How many PTO days do full-time employees get?"))


> 🪄 Three lines. That's the whole library, and it's *the same code you just wrote*. The notebook taught you what each piece does; the package is what you take home.

You can also run it from the terminal:

```bash
python main.py ingest
python main.py ask "How many PTO days do full-time employees get?"
```


## 16 — Your Turn 🧪

The `corpus/` folder is **pluggable** — drop any `.md` or `.txt` files (or whole subfolders) into it, re-run `ingest`, and ask away.

### Things to try

- 📂 Drop your own notes / docs / a markdown export from your wiki into `corpus/` and ask questions.
- ✂️ In `mini_rag/chunk.py`, change `size` and `overlap`. Re-ingest and see how answer quality shifts.
- 🌡️ In `mini_rag/rag.py`, raise the `temperature`. Watch grounded answers start to drift.
- 🎭 Edit the system prompt in `rag.py` to require **citations** (e.g., *"always quote the exact phrase from the context"*).
- 🔢 Change `top_k` in `ask()` — does retrieving more chunks help or hurt?
- 🧠 Try a question whose answer **isn't** in your corpus. Does the model say "I don't know" — or hallucinate?

### Recap of today

| Concept | Section | What you did |
|---|---|---|
| Tokens | 3 | Encoded text with `tiktoken` |
| Next-token prediction | 4 | Streamed an LLM response |
| Temperature | 5 | Compared 0 vs 1.0 |
| System prompts | 6 | Switched personas |
| Few-shot | 7 | Taught a pattern with 2 examples |
| Chain-of-thought | 8 | Asked for step-by-step reasoning |
| The hallucination | 9 | Watched the model invent a policy |
| Embeddings | 10 | Turned words into vectors |
| Cosine similarity | 11 | Measured meaning |
| Chunking | 12 | Wrote `chunk_text` from scratch |
| ChromaDB | 13 | Stored + queried vectors |
| Mini RAG | 14 | Wired it all together |
| Project | 15 | Same code, organized for reuse |

> 🚀 **Next session:** we'll go beyond simple retrieval — re-ranking, agentic patterns, and tool use.
